<a href="https://colab.research.google.com/github/Shubham4521/Cost-Aware-LLM-Router/blob/main/notebooks/03_rule_router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, re, random

OUT = "/content/drive/MyDrive/llm_router"
GITHUB_USER = "Shubham4521"

df = pd.read_csv(
    f"https://raw.githubusercontent.com/{GITHUB_USER}/cost-aware-llm-router/main/data/eval_set.csv",
    keep_default_na=False)
df["needs_search"] = df["needs_search"].astype(str) == "True"

s = pd.read_csv(f"{OUT}/scored_results.csv", keep_default_na=False)
for c in ["score", "cost_usd", "latency_s"]:
    s[c] = s[c].astype(float)
s["passed"] = s["score"] >= 7
lookup = s.set_index(["id", "tier"])
TIERS = ["free", "cheap", "premium"]

print("Prompts:", len(df), "| Scored rows:", len(s))

Mounted at /content/drive
Prompts: 165 | Scored rows: 495


In [2]:
def evaluate(name, choices, flagged_search=None):
    passed, cost, lat = 0, 0.0, 0.0
    for _, row in df.iterrows():
        r = lookup.loc[(row["id"], choices[row["id"]])]
        ok = bool(r["passed"])
        # if a search prompt is NOT detected, the model gets no search results -> counts as fail
        if flagged_search is not None and row["needs_search"] and not flagged_search[row["id"]]:
            ok = False
        passed += ok
        cost += r["cost_usd"]
        lat += r["latency_s"]
    n = len(df)
    return {"router": name, "pass_rate": round(passed / n, 4),
            "total_cost_usd": round(cost, 4), "avg_latency_s": round(lat / n, 2)}

results = []
for t in TIERS:
    results.append(evaluate(f"always_{t}", {i: t for i in df["id"]}))

random.seed(0)
results.append(evaluate("random", {i: random.choice(TIERS) for i in df["id"]}))

oracle = {}
for i in df["id"]:
    ok = [t for t in TIERS if lookup.loc[(i, t)]["passed"]]
    oracle[i] = ok[0] if ok else "free"
results.append(evaluate("oracle (ceiling)", oracle))

print("Baselines ready:", len(results))

Baselines ready: 5


In [3]:
SEARCH_RE = re.compile(
    r"\b(today|today's|tonight|latest|most recent|this (week|month|year)|"
    r"currently|current|right now|news|headlines|price of|exchange rate|weather|forecast)\b",
    re.I)

def needs_search(prompt):
    return bool(SEARCH_RE.search(prompt))

flag = {r["id"]: needs_search(r["prompt"]) for _, r in df.iterrows()}

caught = sum(flag[i] for i in df[df["needs_search"]]["id"])
false_alarms = df[(~df["needs_search"]) & (df["id"].map(flag))]

print(f"Search prompts caught: {caught} / {df['needs_search'].sum()}")
print(f"False alarms (non-search prompts flagged): {len(false_alarms)}")
for _, r in false_alarms.iterrows():
    print("  ", r["id"], "|", r["prompt"][:90].replace("\n", " "))

Search prompts caught: 15 / 15
False alarms (non-search prompts flagged): 6
   factual_017 | Thin laminations are used in a machine in order to reduce  A. Eddy current losses B. Hyste
   factual_021 | Which of the following changes to a circuit will always bring about an increase in the cur
   reasoning_003 | Today, almost all cars have seat belts. How does improving the design of seat belts help p
   reasoning_008 | The theory of plate tectonics provides the current explanation for the way in which  A. st
   reasoning_014 | What event occurs because of the rotation of Earth?  A. tides B. four seasons C. night and
   reasoning_027 | A scientist was digging for dinosaur bones in a dry river bed. It started to rain and the 


In [4]:
def rule_router(search_tier):
    return {i: (search_tier if flag[i] else "free") for i in df["id"]}

results.append(evaluate("rules: search->cheap", rule_router("cheap"), flag))
results.append(evaluate("rules: search->premium", rule_router("premium"), flag))

res = pd.DataFrame(results)
p = res[res["router"] == "always_premium"].iloc[0]
res["quality_vs_premium_%"] = (res["pass_rate"] / p["pass_rate"] * 100).round(1)
res["cost_vs_premium_%"] = (res["total_cost_usd"] / p["total_cost_usd"] * 100).round(1)

print(res.to_string(index=False))
res.to_csv(f"{OUT}/phase3_rule_results.csv", index=False)

                router  pass_rate  total_cost_usd  avg_latency_s  quality_vs_premium_%  cost_vs_premium_%
           always_free     0.8970          0.0000           1.33                  96.7                0.0
          always_cheap     0.9152          0.1947           8.64                  98.7               42.8
        always_premium     0.9273          0.4546           3.29                 100.0              100.0
                random     0.8909          0.2168           4.23                  96.1               47.7
      oracle (ceiling)     0.9576          0.0266           1.64                 103.3                5.9
  rules: search->cheap     0.9212          0.0235           2.00                  99.3                5.2
rules: search->premium     0.9273          0.1262           1.71                 100.0               27.8


In [5]:
pm = s[(s["tier"] == "premium") & (s["id"].str.startswith("math")) & (~s["passed"])]
ref = df.set_index("id")["reference"]
for _, r in pm.iterrows():
    print("=" * 80)
    print(r["id"], "| reference:", ref[r["id"]], "| output tokens:", r["output_tokens"])
    print("LAST 300 CHARS:", str(r["answer"])[-300:])

math_002 | reference: 28 | output tokens: 160
LAST 300 CHARS: ANSWER: 62
math_018 | reference: 4210_{5} | output tokens: 164
LAST 300 CHARS: ANSWER: 42100_5
math_022 | reference: \frac{1997}{2} | output tokens: 288
LAST 300 CHARS: ANSWER: 998
